# Curated script 
Day 3
* Privilege escalation: Meterpreter get system using named pipe impersonation
* Discovery: these need to be done from original process 
    * arp scan on 142.20.56.0/22
    * discover all installed applications
    * identify domain controller
    * identify any shares on host
* Persistense
    * Migrate to lsass.exe
* Credential access
    * credential dumping using mimikatz 
* Script written to C:\Windows\TEMP\myHbYXTpViwX.vbx. Installed Autorun at   HKLM\Software\Microsoft\Windows\CurrentVersion\Run\RTqWaEHv
* get_gui to add administrator "admin" to administrators and RDP group (RDP)    

In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
import networkx as nx

import re
import pickle as pkl

In [2]:
# get directory list 
myFiles = []
def absoluteFilePaths(directory):
    for dirpath,_,filenames in os.walk(directory):
        for f in filenames:
            if bool(re.search('.*graph_\d+', f)):
                myFiles.append(os.path.abspath(os.path.join(dirpath, f)))
absoluteFilePaths("/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/")
myFiles = sorted(myFiles)
for file in myFiles:
    print(file)

/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/graph_00
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/graph_01
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/graph_02
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/graph_03
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/graph_04
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/graph_05
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/graph_06
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/graph_07
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/graph_08
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/graph_09
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/graph_10
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/graph_11
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/graph_12
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/graph_13
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/graph_14
/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/graph_15
/mnt/raid0_ssd_8tb/Bibek

In [ ]:
# test edges
# G = nx.MultiDiGraph()
# G = nx.read_gpickle(myFiles[0])
# print(len(G.nodes))
# print(len(G.edges))
# for node in G.nodes:
#     print(G.nodes[node]["oType"])
#   #  print(G[node])
#     break
# #print(G.adj["3fb3043-fa23-44f9-9ecd-03185550fb63")
# for edge in G.edges:
#     print(edge[0], edge[1], edge[2])
#     print(G.edges[edge]['attributes'])
#     break

In [3]:
G = nx.MultiDiGraph()
#pid2actorId = {}

# subject, event, object triplets 
fileName = "/mnt/raid0_ssd_8tb/Bibek/NCR/ecar/evaluation/25Sept/AIA-051-075/AIA-51-75.ecar-last.json"

# first pass scan the nodes, and add them
# Create the process treee
with open(fileName) as logFile:
    lCount = 0
    for line in logFile:
        line = line.strip()
        try:
            y = json.loads(line)
        except:
            print(line)
        #right now focus on host 051
        if y["hostname"] != "SysClient0051.systemia.com":
            continue
        
        # get the object and action
#         if y["action"] not in actionPool or y["object"] not in objectPool:
#             continue
        
        # now get the subject and objects 
        actorId = y["actorID"]
        pid = y["pid"]
        ppid = y["ppid"]
        objectId = y["objectID"]
        objectType = y["object"]
        G.add_node(actorId)
        G.add_node(objectId)
        G.nodes[actorId]["pid"] = pid
        G.nodes[actorId]["ppid"] = pid
#        G.nodes[actorId]["oType"] = "PROCESS"
        
        G.nodes[objectId]["oType"] = objectType
        
        properties = y["properties"]
        properties["action"] = y["action"]
        properties["principal"] = y["principal"]
        properties["hostname"] = y["hostname"]
        properties["timestamp"] = y["timestamp"]
        #[key = value for key,value in enum(properties)]
        G.add_edge(y["actorID"], y["objectID"], attributes = properties)
        
        #processTree.add_edge(ppid, pid)

In [1]:
print(len(G.nodes))
for node in G.nodes:
    print(G.nodes[node]["oType"])
  #  print(G[node])
    break
#print(G.adj["3fb3043-fa23-44f9-9ecd-03185550fb63")
for edge in G.edges:
    print(edge[0], edge[1], edge[2])
    print(G.edges[edge]['attributes'])
    break

NameError: name 'G' is not defined

In [137]:
def searchExec(T0000_proc, T0000_events):
    T0000_e1 = []
    T0000_e2 = []
    T0000_e3 = []
    
    # Search for file downloads 
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        #print(attributes)
        if G.nodes[v]["oType"] == "FILE" and attributes['action'] == "CREATE":
            try:
                file_path = attributes['file_path']
                if "\\update.exe" in file_path:
                    T0000_proc.add(u)
                    T0000_e1.append((u, v, attributes))
            except:
                pass    
    
    T0000_events.append(T0000_e1)
    
    # search for the file executions 
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        #print(attributes)
        if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
            try:
                image_path = attributes['image_path']
                if "\\update.exe" in image_path:
                    T0000_proc.add(u)
                    T0000_e2.append((u, v, attributes))
            except:
                pass    
    
    T0000_events.append(T0000_e2)
    
    # The executed file starts a reverse shell and starts messaging client
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        #print(attributes)
        if G.nodes[v]["oType"] == "FLOW" and attributes['action'] == "MESSAGE":
            try:
                image_path = attributes['image_path']
                if "\\update.exe" in image_path:
                    T0000_proc.add(u)
                    T0000_e3.append((u, v, attributes))
            except:
                pass    
    
    T0000_events.append(T0000_e3)
    

In [37]:
# meterpreter getsystem : named pipe impersonation 
def searchT0000(T0000_proc, T0000_events):
    T0000_e1 = []
    T0000_e2 = []
    T0000_e3 = []
    
    # "rundll" or "DevSatCookie" started by SYSTEM that started with NETWORK SERVICE or LOCAL SERVICE
    # need to modify graph - the principal right now is stored on edges 

    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        #print(attributes)
        if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
            try:
                command_line = attributes['command_line']
                if "rundll" in command_line or "DevSatCookie" in command_line:
                    if "SYSTEM" in attribute['principal']:
                        #print(attributes['command_line'])
                        T0000_proc.add(u)
                        T0000_e1.append((u, v, attributes))
            except:
                pass    
    
    T0000_events.append(T0000_e1)
    
    # cmd or powersghell started by unusual parents 
    image_dict = ["cmd.exe", "powershell.exe", "wscript.exe", "cscript.exe"]
    parent_dict = ["httpd.exe", "sqlserver.exe", "jbosssvc.exe", "w3wp.exe", "nginx.exe", "php-cgi.exe", "tomcat8.exe", "tomcat7.exe", "tomcat6.exe", "tomcat5.exe", "tomcat.exe"]
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        #print(attributes)
        if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
            try:
                image_path = attributes['image_path']
                parent_path = attributes['parent_image_path']
                
                isImage = False
                for img in image_dict:
                    if img in image_path:
                        isImage = True
                
                isParent = False
                for img in parent_dict:
                    if img in parent_path:
                        isParent = True
                
                if isImage and isParent:
                    T0000_proc.add(u)
                    T0000_e2.append((u, v, attributes))
            except:
                pass    
    
    T0000_events.append(T0000_e2)
    
    # cobalt/ meterpreter getsystem 
    # parent_image = services.exe
    command_dict = ["cmd.*echo.*pipe", "rundll.*dll"]
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        #print(attributes)
        if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
            try:
                parent_path = attributes['parent_image_path']
                command_line = attributes['command_line']
                if "services.exe" in parent_path:
                    if bool(re.search("cmd.*echo.*pipe", command_line)) or bool(re.search("rundll.*dll", command_line)):
                        T0000_proc.add(u)
                        T0000_e3.append((u, v, attributes))
            except:
                pass    
    
    T0000_events.append(T0000_e3)

In [12]:
# Discovery: these need to be done from original process
# arp scan on 142.20.56.0/22
def searchT1016(T1016_proc, T1016_events):
    cmd_dict = ["net ", "ipconfig", "arp ", "netsh ", "nbtstat ", "ping "]
    T1016_e2 = []
    T1016_e1 = []
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        #print(attributes)
        if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
            try:
                command_line = attributes['command_line']
                for cmd in cmd_dict:
                    if cmd in command_line:
                        #print(attributes['command_line'])
                        T1016_proc.add(u)
                        T1016_e1.append((u, v, attributes))
            except:
                pass
                        
    T1016_events.append(T1016_e1)
    
    for u in T1016_proc:
        for dst, attrs in G[u].items():
            v = dst
            for key, attr in attrs.items():
                if G.nodes[v]["oType"] == "FLOW" and attr['attributes']['action'] == "MESSAGE":
                    #T1016_proc.add(item[0])
                    T1016_e2.append((u, v, attr))

    T1016_events.append(T1016_e2)

#     print(len(T1016_e1))
#     print(len(T1016_e2))
#     print(len(T1016_proc))
    

In [14]:
# discover all installed applications TODO
# identify domain controller
# T1087 Account Discovery
# PROCESS --START-> PROCESS 
# - (image_dir="net.exe" OR "powershell.exe") AND 
# - (process_command_line="*net* user*" OR "*net* group*" OR "*net* localgroup*" OR "cmdkey*\/list*" OR 
#   "*get-localuser*" OR "*get-localgroupmembers*" OR "*get-aduser*" OR "query*user*")
def searchT1087(T1087_proc, T1087_events):
    T1087_e1 = []
    image_dict = ["net.exe", "powershell.exe"]
    cmd_dict = ["net.* user", "net.* group", "net.* localgroup", "cmdkey.*\/list", "get-localuser", "get-localgroupmembers", "get-aduser", "query.*user"]

    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        #print(attributes)
        if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
            isImage = False
            try:
                image_path = attributes['image_path']
                for img in image_dict:
                    if img in image_path:
                        isImage = True
                        #print(attributes['command_line'])
            except:
                pass
                #print(attributes)

            isCmd = False
            try:
                command_line = re.escape(attributes['command_line'])
                #print(attributes)
                for cmd in cmd_dict:
                    if bool(re.search(cmd, command_line)):
                        isCmd = True
                        #print(attributes['command_line'])
                        #T1087.append((u, v, attributes))
            except:
                pass

            if isCmd and isImage:
                T1087_proc.add(u)
                T1087_e1.append((u, v, attributes))

    T1087_events.append(T1087_e1)

In [16]:
# identify any shares on host
# T1135- Network Share Discovery
# event1 - OPEN PROCESS 
# event2 - MESSSAGE FLOW
# where image_path contains Net.exe AND
# command_line contains "*net* view*" OR "*net* share*" OR "*get-smbshare -Name*"
def searchT1135(T1135_proc, T1135_events):
    T1135_e1 = []
    command_dict = ["net ", "*net* view*", "*net* share*", "*get-smbshare -Name*"]
    
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        #print(attributes)
        if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
            try:
                command_line = attributes['command_line']
                image_path = attributes['image_path']
                if "net.exe" in image_path:
                    for cmd in command_dict:
                        if cmd in command_line:
                            T1135_proc.add(u)
                            T1135_e1.append((u, v, attributes))
            except:
                pass
    
    T1135_events.append(T1135_e1)

In [6]:
# T1003 Credential DUmping 
# Event 1
# Sysmon Event ID = 10
# target_process_path = "C:\\Windows\\system32\\lsass.exe" AND
# process_granted_access = [0x1010 OR 0x1410 OR 0x147a OR 0x143a]
# process_call_trace = "C:\\Windows\\SYSTEM32\\ntdll.dll\* OR C:\\Windows\\system32\\KERNELBASE.dll* OR UNKNOWN(*)"
# NA: doesn't apply to ecar data format 

# Event 2
# Sysmon Event ID = 12 OR 13 OR 14
# process_path != "C:\\WINDOWS\\system32\\lsass.exe"
# registry_key_path = "*\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Authentication\\Credential Provider\\*" OR 
#                    "*\\SYSTEM\\CurrentControlSet\\Control\\Lsa\\*" OR
#                    "*\\SYSTEM\\CurrentControlSet\\Control\\SecurityProviders\\SecurityProviders\\*" OR
#                    "*\\Control\\SecurityProviders\\WDigest\\*" AND 
# registry_key_path != "*\\Lsa\\RestrictRemoteSamEventThrottlingWindow"
def searchT1003(T1003_proc, T1003_events):
    T1003_e1 = []

    regkey_dict = ["\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Authentication\\Credential Provider\\", 
                "\\SYSTEM\\CurrentControlSet\\Control\\Lsa\\", 
                "\\SYSTEM\\CurrentControlSet\\Control\\SecurityProviders\\SecurityProviders\\",
                "\\Control\\SecurityProviders\\WDigest\\"]
    regkey_excl = "\\Lsa\\RestrictRemoteSamEventThrottlingWindow"
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        if G.nodes[v]["oType"] == "REGISTRY":
            try:
                #print(attributes)
                registry_key_path = re.escape(attributes['key'])
                #print(registry_key_path)
                for mykey in regkey_dict:
                    if mykey in registry_key_path and regkey_excl not in registry_key_path:
                        T1003_proc.add(u)
                        T1003_e1.append((u,v,attributes))
            except:
                pass

    #print(len(T1003_proc))
    #print(len(T1003_e1))


    # Event 3
    # Sysmon Event ID = 7
    # driver_loaded = "C:\\Windows\\System32\\samlib.dll" OR
    #                 "C:\\Windows\\System32\\WinSCard.dll" OR
    #                 "C:\\Windows\\System32\\cryptdll.dll" OR
    #                 "C:\\Windows\\System32\\hid.dll" OR
    #                 "C:\\Windows\\System32\\vaultcli.dll"
    # process_path != "*\\Sysmon.exe" AND process_path!="*\\svchost.exe" AND process_path!="*\\logonui.exe"

    module_dict = ["samlib.dll", "WinSCard.dll", "cryptdll.dll", "hid.dll", "vaultcli.dll"]
    image_dict = ["Sysmon.exe", "svchost.exe", "logonui.exe"]

    T1003_e2 = []
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        if G.nodes[v]["oType"] == "MODULE" and attributes['action'] == "LOAD":
            try:
                #print(attributes)
                module_path = attributes['module_path']
                image_path = attributes['image_path']
                isModule = False
                for mydll in module_dict:
                    if mydll in module_path:
                        isModule = True

                for img in image_dict:
                    if img in image_path:
                        isModule = False
                        break
                if isModule:
                    T1003_proc.add(u)
                    T1003_e2.append((u,v,attributes))
            except:
                pass

    #print(len(T1003_proc))
    #print(len(T1003_e2))


    # Event 4
    # Sysmon Event ID = 1
    # Windows Security Event ID = 4688
    # process_name=reg.exe OR
    # process_command_line = "*save*HKLM\\sam*" OR "*save*HKLM\\system*"
    cmd_reg = ["save.*HKLM.*sam", "save.*HKLM.*system"]
    T1003_e3 = []

    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
            try:
                #print(attributes)
                command_line = attributes['command_line']
                image_path = attributes['image_path']
                isImage = False
                if "reg.exe" in image_path:
                    isImage = True
                isCommand = False
                for cmd in cmd_reg:
                    if bool(re.search(cmd, command_line)):
                        #print(attributes)
                        isCommand = True
                if isImage and isCommand:
                    T1003_proc.add(u)
                    T1003_e3.append((u, v, attributes))

            except:
                pass

    #print(len(T1003_proc))
    #print(len(T1003_e3))


    # Event 5
    # Sysmon Event ID = 1
    # Windows Security Event ID = 4688
    # process_command_line = "*Invoke-Mimikatz -DumpCreds*" OR "gsecdump* -a" OR "wce* -o" OR 
    #                        "procdump* -ma lsass.exe*" OR "ntdsutil*ac i ntds*ifm*create full"
    cmd_reg = ["Involve-Mimikatz -DumpCreds.*", "gsecdump.*-a", "wce.*-o", "procdump.*-ma lsass.exe", "ntdsutil.*ac i ntds.*ifm.*create full"]
    T1003_e4 = []

    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
            try:
                #print(attributes)
                command_line = attributes['command_line']

                isCommand = False
                for cmd in cmd_reg:
                    if bool(re.search(cmd, command_line)):
                        #print(attributes)
                        isCommand = True
                if isCommand:
                    T1003_proc.add(u)
                    T1003_e4.append((u, v, attributes))

            except:
                pass

    T1003_events.append(T1003_e1)
    T1003_events.append(T1003_e2)
    T1003_events.append(T1003_e3)
    T1003_events.append(T1003_e4)
    
#     print(len(T1003_proc))
#     print(len(T1003_e1))
#     print(len(T1003_e2))
#     print(len(T1003_e3))
#     print(len(T1003_e4))

In [120]:
# T1060: Registry Run Keys or Start Folder: Persistence Script 
# TODO: fix for startup folders 
# Script written to C:\Windows\TEMP\myHbYXTpViwX.vbs
# Installed Autorun at HKLM\Software\Microsoft\Windows\CurrentVersion\Run\RTqWaEHv
# But it entails 
    # 1. REGISTRY ADD event - add the registry key to run folder
    # 2. file create event - write vbx script to temp folder 
def searchT1060(T1060_proc, T1060_events):
    T1060_e1 = []
    T1060_e2 = []
    # \\REGISTRY\\MACHINE\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Run\\RTqWaEHv
    mykey = "\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Run\\"
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        #print(attributes)
        if G.nodes[v]["oType"] == "REGISTRY":# and attributes['action'] == "ADD":
            try:
                registry_path = attributes['key']
                #print(registry_path)
                if mykey in registry_path:
                    T1060_proc.add(u)
                    T1060_e1.append((u, v, attributes))
            except:
                pass
    
    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        #print(attributes)
        if G.nodes[v]["oType"] == "FILE" and attributes['action'] == "CREATE":
            try:
                file_path = attributes['file_path']
                #print(file_path) \Windows\Temp\myHbYXTpViwX.vbs 
                if bool(re.search(r"\\Windows\\Temp\\.*\.vbs", file_path)):
                    print(file_path)
                    T1060_proc.add(u)
                    T1060_e2.append((u, v, attributes))
            except:
                pass
    
    T1060_events.append(T1060_e1)
    T1060_events.append(T1060_e2)

In [7]:
# T1076 Remote Desktop Protocol

#Event 1
#Sysmon Event ID = 1
#Windows Security Event ID = 4688
#process_name = "tscon.exe" OR process_name="mstsc.exe"

#Event 2
#Sysmon Event ID = 3
#process_path = "*\\tscon.exe" OR process_name="mstsc.exe" OR dst_port=3389 
#initiated = true

#Event 3
#Sysmon Event_id = 12 OR 13 OR 14
#process_path = "C:\\Windows\\system32\\LogonUI.exe" OR
#registry_key_path = "*\\SOFTWARE\\Policies\\Microsoft\\Windows NT\\Terminal Services\\*"
def searchT1076(T1076_proc, T1076_events):
    T1076_e1 = []
    T1076_e2 = []
    T1076_e3 = []

    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        #print(attributes)
        if G.nodes[v]["oType"] == "PROCESS" and attributes['action'] == "OPEN":
            try:
                image_path = attributes['image_path']
                if "tscon.exe" in image_path or "mstsc.exe" in image_path:
                    T1076_proc.add(u)
                    T1076_e1.append((u, v, attributes))
            except:
                pass

    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        #print(attributes)
        if G.nodes[v]["oType"] == "FLOW" and attributes['action'] == "MESSAGE":
            try:
                image_path = attributes['image_path']
                dst_port = attributes['dest_port']
                if "tscon.exe" in image_path or "mstsc.exe" in image_path or dest_port == '3389':
                    T1076_proc.add(u)
                    T1076_e2.append((u, v, attributes))
            except:
                pass

    for edge in G.edges:
        u = edge[0]
        v = edge[1]
        attributes = G.edges[edge]['attributes']
        #print(attributes)
        regActions = ['ADD', 'EDIT', 'REMOVE']
        if G.nodes[v]["oType"] == "REGISTRY" and attributes['action'] in regActions:
            try:
                image_path = attributes['image_path']
                registry_path = attributes['key']
                if "LogonUI.exe" in image_path or "Terminal Services" in registry_path:
                    T1076_proc.add(u)
                    T1076_e3.append((u, v, attributes))
                    #print(attributes)
            except:
                pass

    T1076_events.append(T1076_e1)
    T1076_events.append(T1076_e2)
    T1076_events.append(T1076_e3)
#     print(len(T1076_proc))
#     print(len(T1076_e1))
#     print(len(T1076_e2))
#     print(len(T1076_e3))

In [121]:
# privesc 
T0000_proc = set()
T0000_events = []
searchT0000(T0000_proc, T0000_events)

#discoveries
T1016_proc = set()
T1016_events = []
searchT1016(T1016_proc, T1016_events)

T1087_proc = set()
T1087_events = []
searchT1087(T1087_proc, T1087_events)

T1135_proc = set()
T1135_events = []
searchT1135(T1135_proc, T1135_events)

# credential access - mimikatz 
T1003_proc = set()
T1003_events = []
searchT1003(T1003_proc, T1003_events)

# persistance 
T1060_proc = set()
T1060_events = []
searchT1060(T1060_proc, T1060_events)

T1076_proc = set()
T1076_events = []
searchT1076(T1076_proc, T1076_events)

\Device\HarddiskVolume1\Windows\Temp\myHbYXTpViwX.vbs
\Device\HarddiskVolume1\Windows\Temp\myHbYXTpViwX.vbs


In [141]:
# Execution 
autorun_proc = set()
autorun_events = []
searchExec (autorun_proc, autorun_events)

In [143]:
# Download update.exe, execute it, establish the reverse shell connection
print("Execs")
print(len(autorun_proc))
for events in autorun_events:
    print(len(events))
    for event in events:
        print(event)

Execs
4
7
('3360bc38-784d-422d-9899-779f0e558da9', 'e19df24b-4bc0-4a0e-87ce-85952cfc5ba6', {'acuity_level': '3', 'file_path': '\\Device\\HarddiskVolume1\\Users\\dcoombes\\Downloads\\update.exe', 'image_path': '\\\\?\\C:\\Program Files (x86)\\Mozilla Firefox\\firefox.exe', 'action': 'CREATE', 'principal': 'SYSTEMIACOM\\dcoombes', 'hostname': 'SysClient0051.systemia.com', 'timestamp': '2019-09-25T14:22:59.658-04:00'})
('3360bc38-784d-422d-9899-779f0e558da9', '982464a2-4263-4c78-8aad-bec16a8d651e', {'acuity_level': '3', 'file_path': '\\Device\\HarddiskVolume1\\Users\\dcoombes\\Downloads\\update.exe:Zone.Identifier', 'image_path': '\\\\?\\C:\\Program Files (x86)\\Mozilla Firefox\\firefox.exe', 'action': 'CREATE', 'principal': 'SYSTEMIACOM\\dcoombes', 'hostname': 'SysClient0051.systemia.com', 'timestamp': '2019-09-25T14:22:59.984-04:00'})
('ef03233d-e8d6-49ed-a5e5-8fc71092e5be', 'ee0373a6-6bb1-42ed-a488-db530a38f75e', {'acuity_level': '3', 'file_path': '\\Device\\HarddiskVolume1\\Users\\dco

In [123]:
# Net scan 
print("T1016")
print(len(T1016_proc))
for events in T1016_events:
    print(len(events))

T1016
43
185
729


In [131]:
for proc in T1016_proc:
    print(proc)

ffdd2745-a6f9-479b-b49b-2bf52ad23e9e
27aec587-671d-4e68-ac48-946d55936df3
5d8cbdf6-3358-4fa7-beb3-63271b7d53fa
4800e094-8a92-4de5-941d-1b4e668ccb5b
8914f292-fcea-4e50-baf4-63b93adb8d3a
1d5eed31-4324-4478-bec8-2a522745e041
13fdaf32-75db-4624-af24-d12628cb6726
9528a59a-baef-4622-b469-9539e658d5c5
1f7fb1a4-fea3-4737-adb9-ab298eae3efe
bda7d275-8bbf-4c5b-8130-10a95b829a3e
51c22b33-473a-42ca-a608-7622b491d980
415f6d32-8c93-4cd1-9035-f7d5f92dea3b
9a8170a9-795d-4967-935b-54d3819c0ba5
1627d7e0-bd1d-4d69-9a99-4f2e415b9480
985a6eb1-629f-48a4-9f60-9cd298608c1b
a2a19fa2-39b8-40c7-be15-7712bd0e367e
2018ba71-54bb-4be8-aa6c-6ca43bea11f0
51ad2510-33dc-4c8d-a888-0eff8fd00fb9
2d007160-be0d-4072-b2f9-6cdb87dfe701
3cc8e808-03f5-4ce1-b7fe-82583860bd4a
be9b08db-b724-4293-abb5-4ec3028746a4
917f6126-dcb7-4e30-892f-9626ad012d89
da1134b9-2069-4b3e-b653-c3fc75401b14
475ea186-4dcd-417e-8d4c-71676b807177
00bea5c3-bb41-49dc-8e36-a5b9b8898927
02afe9e2-1ba0-4b0b-8d8a-c791d6341397
86714e49-dcd4-433e-8615-6b178abbe241
9

In [124]:
# software/domain discovery
print("T1087")
print(len(T1087_proc))
for events in T1087_events:
    print(len(events))

T1087
4
17


In [125]:
print("T1135")
print(len(T1135_proc))
for events in T1135_events:
    print(len(events))

T1135
3
9


In [126]:
print("T1003")
print(len(T1003_proc))
for events in T1003_events:
    print(len(events))

T1003
87
0
104
0
0


In [127]:
print("T1060")
print(len(T1060_proc))
for events in T1060_events:
    print(len(events))

T1060
1
1
2


In [128]:
print("T1076")
print(len(T1076_proc))
for events in T1076_events:
    print(len(events))

T1076
14
0
0
176
